E-14: the offset sparse-model bound and rate for the ATT (H14-Bnd, H14-Rate; `thm:sparse_dual_rate`, offset version)

Design: `doc/2026-10-07_experiment_registration.md` (parent repository), E-14, with §1.2, §1.3 A-1/A-3/B/C, §1.8 and §1.9. Since U28 the document is a design, not a binding registration; the implementation choices are listed in the parent repository's `docs/spec/02` §2.1 ("E-14 の" rows). One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-14 stage0 12_E14_sparse_offset_rate.ipynb`
- `... E-14 stage0.5 ...`, then `... E-14 stage1 ...` (each after the previous stage is committed).

The design, the replication function and the aggregation are in `grrexp/e14.py`; the pilot and the confirmatory stage run the same aggregation. A solver that raises stops the notebook (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import e14, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(14, STAGE, cells=None if STAGE == "stage0" else e14.CELLS,
                          arms=None if STAGE == "stage0" else [*e14.PENALTIES, "BKL_u0"])
STAGE

'stage0'

# Stage 0: the frozen design recomputed, the outward constants, and the certificate of the planted $\beta^*$ (§1.9)

In [2]:
if STAGE == "stage0":
    s0 = e14.stage0()
    for r in s0["generators"]:
        print(r["generator"], "frozen", r["frozen_ok"], "certified", r["certified"],
              [k for k, v in {**r["frozen_checks"], **r["certificate_checks"]}.items() if not v])
    if not all(r["frozen_ok"] and r["certified"] for r in s0["generators"]):
        raise RuntimeError("the recomputed design does not match the frozen values or is not certified")
    if not s0["bkl_zero_offset_infeasible"]:
        raise RuntimeError("BKL with u_ref = 0 is not infeasible at beta = 0")
    rec.write_stage0("e14_population", s0)
    rec.finalize(R=None, n=None, warnings=0, failures=0)

SQ frozen True certified True []
UKL frozen True certified True []
BP frozen True certified True []
BKL frozen True certified True []


# The workflow after the replications (shared by Stage 0.5 and Stage 1)

Summary per cell x penalty, H14-Bnd (any replication with (E1)-(E3) whose err$^2$ lower bound exceeds the bound squared), the health checks (failure rate per generator x penalty, Holm 0.01; BKL with $u_{\mathrm{ref}}=0$), and H14-Rate with bootstrap family 4 (stream 4): one call per cell of the SN penalty with $n\in\{16000,32000,64000\}$, in the fixed cell order.

In [3]:
def load_population():
    return json.loads((outputs.stage_dir(14, "stage0") / "stage0_e14_population.json").read_text())


def aggregate(raw, seeds):
    summ = e14.summarise(raw)
    bnd = e14.h14_bnd(summ)
    hl = e14.health(raw)
    if hl["bkl_zero_offset"]["verdict"] == "fail":  # deterministic: an implementation error
        raise RuntimeError(f"BKL u_ref = 0 start not infeasible: {hl['bkl_zero_offset']}")
    rate = e14.rate_predictions(raw, seeds)
    summ["h14_bnd"] = json.dumps(bnd)
    summ["h14_rate"] = json.dumps(rate)
    summ["health"] = json.dumps(hl)
    return summ

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

In [4]:
if STAGE == "stage0.5":
    import matplotlib.pyplot as plt

    parallel.configure_worker()
    s0 = load_population()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e14.CELLS):
            t0 = time.perf_counter()
            tasks = e14.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, s0, cells=[ci])
            res = parallel.run_tasks(e14.replicate, tasks, workers)
            e14.summarise(e14.raw_frame(tasks, res))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e14.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    summ = aggregate(raw, Seeds(14, entropy=PILOT_ENTROPY))
    summ.to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e14.tables(S, None)
    plt.close(e14.figure(S, None))
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "penalty", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e14.CELLS[c] for c in counts["cell"]], "arm": counts["penalty"],
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e14.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": list(e14.N_VALUES)},
                 warnings=e14.total_warnings(raw), failures=int((raw["penalty"].isin(e14.PENALTIES) & (raw["status"] != "ok")).sum()),
                 extra={"whole_pilot_aggregation_table_and_figure_seconds": whole_seconds})
    del raw

# Stage 1: confirmatory Monte Carlo

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    s0 = load_population()
    tasks = e14.tasks_for(CONFIRMATORY_ENTROPY, e14.R, s0)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e14.replicate, tasks, outputs.PILOT_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e14.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e14.total_warnings(raw)
    raw.shape, n_warn

In [6]:
if STAGE == "stage1":
    summ = aggregate(raw, Seeds(14))
    reeval = e14.reevaluate(s0, raw)  # N = 1e7 re-evaluation of any H14-Bnd violation (reported only)
    summ["h14_bnd_reevaluation"] = json.dumps(reeval)
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("h14_bnd", "health", "h14_rate")}

# Table and figure (from summary.csv only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    for name, tex in e14.tables(S, None).items():
        rec.write_table(name, tex)
    rec.save_figure(e14.figure(S, None), "fig_E14_rate")
    print(json.loads(S["h14_rate"].iloc[0]))
    rec.finalize(R={"main": e14.R}, n={"main": list(e14.N_VALUES)}, warnings=int(n_warn),
                 failures=int((raw["penalty"].isin(e14.PENALTIES) & (raw["status"] != "ok")).sum()))